<a href="https://colab.research.google.com/github/thanh-huy/SIC_Excercies/blob/main/finetunechapt10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import files

from sklearn.model_selection import train_test_split
from sklearn.metrics import *

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import load_dataset, Dataset


In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [4]:
uploaded = files.upload()
data_path = list(uploaded.keys())[0]


Saving company-document-text.csv to company-document-text.csv


In [5]:
data = pd.read_csv(data_path)
data.head()


,text,label,word_count
0,order id 10718 shipping details ship name k...,ShippingOrder,120
1,invoice order id 10707 customer id arout ord...,invoice,66
2,order id 10448 shipping details ship name r...,ShippingOrder,96
3,invoice order id 11068 customer id queen ord...,invoice,68
4,order id 10656 shipping details ship name g...,ShippingOrder,109


In [8]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
data['label'] = le.fit_transform(data['label'])
data.head()


,text,label,word_count
0,order id 10718 shipping details ship name k...,0,120
1,invoice order id 10707 customer id arout ord...,1,66
2,order id 10448 shipping details ship name r...,0,96
3,invoice order id 11068 customer id queen ord...,1,68
4,order id 10656 shipping details ship name g...,0,109


In [9]:
data2 = data.loc[:, ['text','label']].copy()
train, val = train_test_split(data2, test_size=0.3, random_state=10)


In [10]:
train_ds = Dataset.from_pandas(train)
val_ds = Dataset.from_pandas(val)


In [11]:
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [12]:
def preprocess_function(data):
    return tokenizer(data['text'], padding="max_length", truncation=True, max_length=130)

train_ds = train_ds.map(preprocess_function, batched=True)
val_ds = val_ds.map(preprocess_function, batched=True)


Map:   0%|          | 0/1873 [00:00<?, ? examples/s]

Map:   0%|          | 0/803 [00:00<?, ? examples/s]

In [13]:
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels = 4)


model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [21]:
args = TrainingArguments(
    output_dir = './results',          # where to save checkpoints/outputs
    eval_strategy = "epoch",           # run eval at the end of each epoch
    save_strategy = "epoch",           # save a checkpoint every epoch
    learning_rate = 2e-5,              # base learning rate
    per_device_train_batch_size = 32,  # train batch size per device
    per_device_eval_batch_size = 32,   # eval batch size per device
    num_train_epochs  = 3,             # total training epochs
    weight_decay = 0.02,               # L2 weight decay strength
    load_best_model_at_end = True,     # reload best checkpoint (by metric)
    logging_steps = 10,                # log every N training steps
    report_to = "tensorboard"          # send logs to TensorBoard
)

In [22]:
trainer = Trainer(
    model = model,
    args = args,                  # TrainingArguments
    train_dataset = train_ds,
    eval_dataset = val_ds,
    processing_class = tokenizer,
)
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.095903,0.029158
2,0.012732,0.008349
3,0.009352,0.006591


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

TrainOutput(global_step=177, training_loss=0.14571390730344644, metrics={'train_runtime': 200.7572, 'train_samples_per_second': 27.989, 'train_steps_per_second': 0.882, 'total_flos': 375387077903760.0, 'train_loss': 0.14571390730344644, 'epoch': 3.0})